In [ ]:
# 1. Imports
import sys
import os
from pathlib import Path

def get_project_root():
    if "__file__" not in globals():
        cwd = Path(os.getcwd()).resolve()
        for parent in [cwd] + list(cwd.parents):
            if (parent / "src").exists():
                return parent
        return cwd
    return Path(__file__).resolve().parents[1]

ROOT = get_project_root()
sys.path.insert(0, str(ROOT))
print("Project root added to sys.path:", ROOT)

import numpy as np
import pandas as pd

# Import embedded data modules
from src.likelihoods.data.planck_2015 import PLANCK_2015
from src.likelihoods.data.planck_2018_recon import PLANCK_2018_RECON
from src.likelihoods.data.desi_bao_dr1 import DESI_BAO_DR1
from src.likelihoods.data.cosmic_chronometers import COSMIC_CHRONOMETERS

# Import likelihoods and pipelines
from src.likelihoods.planck_shoes_joint import PlanckSH0ESJointLikelihood
from src.likelihoods.desi_bao import DESIBAO
from src.likelihoods.cosmic_chronometers import CosmicChronometers
from src.likelihoods.joint_likelihood import JointLikelihood
from src.physics.mcmc_joint_pipeline import JointMCMCPipeline
from src.pipeline.paper_figures_pipeline import PaperFiguresPipeline

# 2. Build likelihoods from embedded data
def build_joint_likelihood(planck_dict):
    planck_like = PlanckSH0ESJointLikelihood(planck_dict)
    bao_like = DESIBAO(DESI_BAO_DR1)
    cc_like = CosmicChronometers(COSMIC_CHRONOMETERS)
    return JointLikelihood(planck_like, bao_like, cc_like)

joint_2015 = build_joint_likelihood(PLANCK_2015)
joint_2018 = build_joint_likelihood(PLANCK_2018_RECON)

# 3. Define S.T.A.R. Model
H_expr = "H0*sqrt(Ωm*(1+z)**3 + ΩΛ + a*z + b*z**2)"
param_names = ["H0", "Ωm", "ΩΛ", "a", "b"]

priors = {
    "H0": (70, 5),
    "Ωm": (0.3, 0.1),
    "ΩΛ": (0.7, 0.1),
    "a": (0, 0.1),
    "b": (0, 0.1),
}

proposal_widths = {k: 0.01 for k in param_names}

# 4. Run MCMC for both Planck datasets
pipeline_2015 = JointMCMCPipeline(H_expr, param_names, priors, proposal_widths, joint_2015)
chain_2015 = pipeline_2015.run(theta0=[70, 0.3, 0.7, 0, 0], nsteps=8000)

pipeline_2018 = JointMCMCPipeline(H_expr, param_names, priors, proposal_widths, joint_2018)
chain_2018 = pipeline_2018.run(theta0=[70, 0.3, 0.7, 0, 0], nsteps=8000)

# 5. Generate paper figures for both runs
fig_pipeline_2015 = PaperFiguresPipeline(
    chain_2015,
    param_names,
    H_expr,
    data_paths={"planck": "PLANCK_2015", "bao": "DESI_BAO_DR1", "cc": "COSMIC_CHRONOMETERS"}
)
constraints_2015 = fig_pipeline_2015.run("paper_figures_planck2015")

fig_pipeline_2018 = PaperFiguresPipeline(
    chain_2018,
    param_names,
    H_expr,
    data_paths={"planck": "PLANCK_2018_RECON", "bao": "DESI_BAO_DR1", "cc": "COSMIC_CHRONOMETERS"}
)
constraints_2018 = fig_pipeline_2018.run("paper_figures_planck2018")

constraints_2015, constraints_2018
